In [ ]:
# Constants for test case generation
import os

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "your-api-key-here")
APP_NAME = "MyApp"
APP_PACKAGE_NAME = "com.example.myapp"
APP_CONTEXT = """
A mobile banking app that allows users to view balance, transfer funds,
pay bills, and manage cards. Target: Android and iOS.
"""

# Model used for BDD test generation (e.g. gpt-4o-mini, gpt-4o)
MODEL_NAME = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")

**Setup:** Install the OpenAI client if needed: `pip install openai`. Set `OPENAI_API_KEY` in the constants cell (or in your environment).

In [ ]:
# LLM client and helper for BDD test case generation
import json
import re
from openai import OpenAI

if not OPENAI_API_KEY or OPENAI_API_KEY == "your-api-key-here":
    raise ValueError("Set OPENAI_API_KEY in the constants cell or in your environment.")

client = OpenAI(api_key=OPENAI_API_KEY)

BDD_SCHEMA = """
Return a JSON array of test features. Each feature has:
- "feature": string (feature name)
- "scenarios": array of { "scenario": string, "steps": [ {"type": "Given"|"When"|"Then", "text": string}, ... ] }
Output only valid JSON, no markdown or extra text.
"""


def call_llm_for_bdd(system_prompt: str, user_prompt: str, model: str = None) -> list[dict]:
    """Call OpenAI and parse response into list of BDD feature dicts."""
    if model is None:
        model = MODEL_NAME
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.3,
    )
    text = response.choices[0].message.content.strip()
    # Strip markdown code block if present
    if "```json" in text:
        text = re.search(r"```(?:json)?\s*([\s\S]*?)```", text).group(1).strip()
    elif "```" in text:
        text = re.sub(r"^```\w*\s*", "", text).strip()
        text = re.sub(r"\s*```$", "", text).strip()
    return json.loads(text)

## 1. Simple test case generation (app name only)

Generates BDD-style test cases using only the app name. Saves to `test_cases_simple/`.

In [ ]:
import json
from pathlib import Path

OUTPUT_DIR_SIMPLE = Path("test_cases_simple")
OUTPUT_DIR_SIMPLE.mkdir(exist_ok=True)


def generate_test_cases_simple(app_name: str) -> list[dict]:
    """
    Generate BDD-format test cases based on app name only.
    Returns list of test case dicts with feature, scenario, steps (Given/When/Then).
    """
    system = f"""You are a QA engineer. Generate mobile app test cases in BDD format (Given/When/Then).
{BDD_SCHEMA}"""
    user = f"""Generate 4-6 test scenarios for the mobile app "{app_name}". Cover: launch, navigation, back button, and 1-2 other common flows. Each scenario must have Given, When, Then steps. Output only the JSON array."""
    return call_llm_for_bdd(system, user)


# Generate and save
cases_simple = generate_test_cases_simple(APP_NAME)
out_file = OUTPUT_DIR_SIMPLE / f"{APP_NAME.replace(' ', '_')}_test_cases.json"
with open(out_file, "w") as f:
    json.dump(cases_simple, f, indent=2)
print(f"Saved {len(cases_simple)} feature(s) to {out_file}")

## 2. Test case generation with app name and context

Generates BDD test cases using app name and app context (domain, target platform, etc.). Saves to `test_cases_with_context/`.

In [ ]:
OUTPUT_DIR_CONTEXT = Path("test_cases_with_context")
OUTPUT_DIR_CONTEXT.mkdir(exist_ok=True)


def generate_test_cases_with_context(app_name: str, app_context: str) -> list[dict]:
    """
    Generate BDD-format test cases using app name and context (domain, platform, etc.).
    """
    system = f"""You are a QA engineer. Generate mobile app test cases in BDD format (Given/When/Then) tailored to the app's domain and context.
{BDD_SCHEMA}"""
    user = f"""App: "{app_name}"

Context:
{app_context.strip()}

Generate 5-8 test scenarios that make sense for this app (e.g. onboarding, main flows, edge cases). Each scenario must have Given, When, Then steps. Output only the JSON array."""
    return call_llm_for_bdd(system, user)


# Generate and save
cases_context = generate_test_cases_with_context(APP_NAME, APP_CONTEXT)
out_file = OUTPUT_DIR_CONTEXT / f"{APP_NAME.replace(' ', '_')}_test_cases.json"
with open(out_file, "w") as f:
    json.dump(cases_context, f, indent=2)
print(f"Saved {len(cases_context)} feature(s) to {out_file}")

## 3. Test case generation from app screen graph (nodes = screens, edges = interactions)

Reads a JSON file describing the app as a graph (screens as nodes, interactions as edges) and generates BDD test cases that cover paths through the graph. Saves to `test_cases_from_graph/`.

In [ ]:
# Path to the provided graph JSON (screens = nodes, interactions = edges)
APP_GRAPH_JSON_PATH = Path("app_screen_graph.json")

OUTPUT_DIR_GRAPH = Path("test_cases_from_graph")
OUTPUT_DIR_GRAPH.mkdir(exist_ok=True)

# Example structure of the expected graph JSON (for reference):
# {
#   "nodes": [
#     {"id": "home", "label": "Home Screen", "description": "..."},
#     {"id": "login", "label": "Login Screen", "description": "..."}
#   ],
#   "edges": [
#     {"source": "splash", "target": "login", "action": "auto_navigate"},
#     {"source": "login", "target": "home", "action": "tap_login"}
#   ]
# }


def load_app_graph(graph_path: Path) -> dict:
    """Load app screen graph from JSON. Returns dict with 'nodes' and 'edges'."""
    with open(graph_path) as f:
        return json.load(f)


def generate_test_cases_from_graph(
    app_name: str, graph: dict, graph_path: Path
) -> list[dict]:
    """
    Generate BDD-format test cases from app graph (screens as nodes, interactions as edges).
    Covers key paths and transitions between screens.
    """
    nodes = graph.get("nodes", [])
    edges = graph.get("edges", [])
    if not nodes and not edges:
        return []

    system = f"""You are a QA engineer. Given an app screen graph (nodes = screens, edges = transitions), generate BDD test cases that cover screen flows.
{BDD_SCHEMA}"""
    user = f"""App: "{app_name}"

Screen graph (nodes = screens, edges = user actions between screens):
Nodes: {json.dumps(nodes)}
Edges: {json.dumps(edges)}

Generate BDD test scenarios for the main user flows through these screens. Each scenario should be one path (Given user is on screen X, When they do action Y, Then they see screen Z). Include 1-2 longer multi-step flows. Output only the JSON array."""
    return call_llm_for_bdd(system, user)


# Generate from graph file (create dummy graph if file missing, for demo)
if APP_GRAPH_JSON_PATH.exists():
    graph = load_app_graph(APP_GRAPH_JSON_PATH)
else:
    # Dummy graph for demo when no file is provided
    graph = {
        "nodes": [
            {"id": "splash", "label": "Splash Screen"},
            {"id": "login", "label": "Login Screen"},
            {"id": "home", "label": "Home Screen"},
            {"id": "transfer", "label": "Transfer Screen"},
        ],
        "edges": [
            {"source": "splash", "target": "login", "action": "auto_navigate"},
            {"source": "login", "target": "home", "action": "tap_login"},
            {"source": "home", "target": "transfer", "action": "tap_transfer"},
            {"source": "transfer", "target": "home", "action": "tap_back"},
        ],
    }

cases_graph = generate_test_cases_from_graph(APP_NAME, graph, APP_GRAPH_JSON_PATH)
out_file = OUTPUT_DIR_GRAPH / f"{APP_NAME.replace(' ', '_')}_graph_test_cases.json"
with open(out_file, "w") as f:
    json.dump(cases_graph, f, indent=2)
print(f"Saved {len(cases_graph)} scenario(s) to {out_file}")